In [1]:
suppressPackageStartupMessages({
    library(tidyverse)
    library(kSamples)
    library(glmnet)
    library(ppcor)
    library(Signac)
    library(Seurat)
    library(argparse)
    library(logging)
    library(reshape2)
})

Warning message:
“package ‘tibble’ was built under R version 4.3.2”
Warning message:
“package ‘stringr’ was built under R version 4.3.2”


In [2]:
SINCERITITES <- dget("/home/chenxufeng/WorkSpace/scMagnify/scMagnify-benchmark/scripts/SINCERITIES.R")

In [71]:
path <- "/home/chenxufeng/picb_cxf/Data/scMultiSim/bench_grn/datasets/grn1139_noise_tree1_1000_cells1139_genes_sigma0.1_1/"

In [72]:
ExprMatrix <- read.csv(file = paste0(path , "ExpressionData.csv"), row.names = 1)

In [73]:
head(ExprMatrix)

,cell1,cell2,cell3,cell4,cell5,cell6,cell7,cell8,cell9,cell10,⋯,cell991,cell992,cell993,cell994,cell995,cell996,cell997,cell998,cell999,cell1000
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
gene1,0.000000,0.000000,0.000000,0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,⋯,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000
gene2,0.000000,0.000000,0.000000,0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,⋯,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000
gene3,7.693487,7.011227,6.857981,0,6.087463,7.629357,7.577429,5.426265,7.839204,7.209453,⋯,6.108524,6.022368,6.022368,7.076816,5.392317,5.857981,7.294621,6.714246,6.00000,5.087463
gene4,0.000000,0.000000,0.000000,0,0.000000,0.000000,0.000000,6.426265,0.000000,0.000000,⋯,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000
gene5,4.584963,0.000000,4.087463,0,3.459432,2.584963,5.169925,7.000000,0.000000,0.000000,⋯,4.000000,0.000000,5.426265,0.000000,3.700440,0.000000,5.247928,0.000000,4.70044,6.285402
gene6,5.209453,5.906891,5.491853,0,4.321928,0.000000,0.000000,0.000000,3.459432,5.000000,⋯,0.000000,3.169925,0.000000,0.000000,2.807355,0.000000,4.700440,0.000000,6.00000,0.000000


In [74]:
sds <- apply(ExprMatrix, 2, sd)
ExprMatrix<- ExprMatrix[, which(sds > quantile(sds, prob = 0.2))]

In [75]:
Pseudotime <- read.csv(file = paste0(path , "PseudoTime.csv"))

In [76]:
sorted_indices <- order(Pseudotime$palantir_pseudotime)
bin_indices <- cut(seq_along(sorted_indices), breaks = 10, labels = FALSE) - 1

Pseudotime$time <- NA
Pseudotime$time[sorted_indices] <- bin_indices

In [77]:
preprocess <- function(df){
  
  NAIdx <- is.na(apply(df,1,sum))
  df <- df[!NAIdx,]
  
  totDATA <- df[,1:dim(df)[2]-1]
  timeline <- df[,dim(df)[2]]
  DATA.time <- sort(unique(timeline))
  DATA.num_time_points <- length(DATA.time)
  DATA.totDATA <- matrix(ncol = dim(df)[2]-1)
  DATA.timeline <- vector()
  
  for (k in 1:DATA.num_time_points) {
    I <- which(timeline==DATA.time[k])
    DATA.totDATA <- rbind(DATA.totDATA,as.matrix(totDATA[I,]))
    DATA.timeline <- c(DATA.timeline,timeline[I])
  }
  DATA.totDATA <- DATA.totDATA[-1,]
  DATA.totDATA[is.na(DATA.totDATA)] <- 0
  DATA.numGENES <- dim(DATA.totDATA)[2]
  
  DATA.genes <- colnames(df)[1:dim(df)[2]-1]
  
  DATA.singleCELLdata <- by(DATA.totDATA,DATA.timeline,identity)
  DATA <- list(time=DATA.time,num_time_points=DATA.num_time_points,
               totDATA=DATA.totDATA,timeline=DATA.timeline,numGENES=DATA.numGENES,
               genes=DATA.genes,singleCELLdata=DATA.singleCELLdata)
  return(DATA)
}

In [78]:
DATA <- preprocess(cbind(t(ExprMatrix), Pseudotime$time))

Warning message in cbind(t(ExprMatrix), Pseudotime$time):
“number of rows of result is not a multiple of vector length (arg 2)”


In [79]:
str(DATA)

List of 7
 $ time           : num [1:9] 0 1 2 3 4 5 6 7 8
 $ num_time_points: int 9
 $ totDATA        : num [1:800, 1:1250] 0 0 0 0 0 0 0 0 0 0 ...
  ..- attr(*, "dimnames")=List of 2
  .. ..$ : chr [1:800] "cell2" "cell3" "cell4" "cell5" ...
  .. ..$ : chr [1:1250] "gene1" "gene2" "gene3" "gene4" ...
 $ timeline       : Named num [1:800] 0 0 0 0 0 0 0 0 0 0 ...
  ..- attr(*, "names")= chr [1:800] "cell2" "cell3" "cell4" "cell5" ...
 $ numGENES       : int 1250
 $ genes          : chr [1:1250] "gene1" "gene2" "gene3" "gene4" ...
 $ singleCELLdata :List of 9
  ..$ 0:'data.frame':	100 obs. of  1250 variables:
  .. ..$ gene1   : num [1:100] 0 0 0 0 0 0 0 0 0 0 ...
  .. ..$ gene2   : num [1:100] 0 0 0 0 0 0 0 0 0 0 ...
  .. ..$ gene3   : num [1:100] 7.01 6.86 0 6.09 7.63 ...
  .. ..$ gene4   : num [1:100] 0 0 0 0 0 ...
  .. ..$ gene5   : num [1:100] 0 4.09 0 3.46 2.58 ...
  .. ..$ gene6   : num [1:100] 5.91 5.49 0 4.32 0 ...
  .. ..$ gene7   : num [1:100] 0 0 4.91 6.61 4.95 ...
  .. ..$ ge

In [80]:
SIGN <- 0
result <- SINCERITITES(DATA, distance=1, method = 1, noDIAG = 0, SIGN = SIGN)

Warning message in ks.test.default(p1, p2):
“p-value will be approximate in the presence of ties”
Warning message in ks.test.default(p1, p2):
“p-value will be approximate in the presence of ties”
Warning message in ks.test.default(p1, p2):
“p-value will be approximate in the presence of ties”
Warning message in ks.test.default(p1, p2):
“p-value will be approximate in the presence of ties”
Warning message in ks.test.default(p1, p2):
“p-value will be approximate in the presence of ties”
Warning message in ks.test.default(p1, p2):
“p-value will be approximate in the presence of ties”
Warning message in ks.test.default(p1, p2):
“p-value will be approximate in the presence of ties”
Warning message in ks.test.default(p1, p2):
“p-value will be approximate in the presence of ties”
Warning message in ks.test.default(p1, p2):
“p-value will be approximate in the presence of ties”
Warning message in ks.test.default(p1, p2):
“p-value will be approximate in the presence of ties”
Warning message in k

In [81]:
adj_matrix <- result$adj_matrix

In [82]:
dimnames(adj_matrix) <- list(DATA$genes,DATA$genes)

In [83]:
edge_df <- melt(adj_matrix, varnames = c("TF", "Target"), value.name = "Score")

In [84]:
edge_df

TF,Target,Score
<fct>,<fct>,<dbl>
gene1,gene1,0.000000e+00
gene2,gene1,2.603028e-37
gene3,gene1,5.944476e-38
gene4,gene1,0.000000e+00
gene5,gene1,2.364009e-37
gene6,gene1,4.228377e-38
gene7,gene1,0.000000e+00
gene8,gene1,0.000000e+00
gene9,gene1,3.180317e-38


In [85]:
write.csv(edge_df, file = file.path(path, paste0("SINCERITIES.csv")), row.names = F)